In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
path_to_data = "dataset/weatherAUS.csv"
df = pd.read_csv(path_to_data)
df

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145455,2017-06-21,Uluru,2.8,23.4,0.0,NaN,NaN,E,31.0,SE,...,51.0,24.0,1024.6,1020.3,NaN,NaN,10.1,22.4,No,No
145456,2017-06-22,Uluru,3.6,25.3,0.0,NaN,NaN,NNW,22.0,SE,...,56.0,21.0,1023.5,1019.1,NaN,NaN,10.9,24.5,No,No
145457,2017-06-23,Uluru,5.4,26.9,0.0,NaN,NaN,N,37.0,SE,...,53.0,24.0,1021.0,1016.8,NaN,NaN,12.5,26.1,No,No
145458,2017-06-24,Uluru,7.8,27.0,0.0,NaN,NaN,SE,28.0,SSE,...,51.0,24.0,1019.4,1016.5,3.0,2.0,15.1,26.0,No,No


In [3]:
round(100*df.isna().sum()/len(df),1)

Date              0.0
Location          0.0
MinTemp           1.0
MaxTemp           0.9
Rainfall          2.2
Evaporation      43.2
Sunshine         48.0
WindGustDir       7.1
WindGustSpeed     7.1
WindDir9am        7.3
WindDir3pm        2.9
WindSpeed9am      1.2
WindSpeed3pm      2.1
Humidity9am       1.8
Humidity3pm       3.1
Pressure9am      10.4
Pressure3pm      10.3
Cloud9am         38.4
Cloud3pm         40.8
Temp9am           1.2
Temp3pm           2.5
RainToday         2.2
RainTomorrow      2.2
dtype: float64

In step 3 : Evaporation ,  Cloud9am , Cloud3pm, Evaporation, Sunshine were removed since a lot of missinge data
But XGBoost natively supports NaN , no benefit to reomoving the column away,  Leaving them in.

In [4]:
# drop_column = ["Evaporation","Sunshine","Cloud9am","Cloud3pm"]
# df=df.drop(drop_column,axis=1)
# round(100*df.isna().sum()/len(df),1)

# TARGETS

Drop nan rows for target data

In [5]:
#target data def 
target_name = ['RainTomorrow','Rainfall']
df=df.dropna(subset=target_name).reset_index(drop=True)
round(100*df.isna().sum()/len(df),1)

Date              0.0
Location          0.0
MinTemp           0.3
MaxTemp           0.2
Rainfall          0.0
Evaporation      42.4
Sunshine         47.5
WindGustDir       6.5
WindGustSpeed     6.5
WindDir9am        6.9
WindDir3pm        2.6
WindSpeed9am      0.7
WindSpeed3pm      1.8
Humidity9am       1.1
Humidity3pm       2.5
Pressure9am       9.8
Pressure3pm       9.8
Cloud9am         37.4
Cloud3pm         39.8
Temp9am           0.5
Temp3pm           1.9
RainToday         0.0
RainTomorrow      0.0
dtype: float64

In [6]:
#Replace Raintoday and RainTomorrow by binary values
df['RainToday']=df['RainToday'].map({'No':0 , 'Yes':1})
df['RainTomorrow']=df['RainTomorrow'].map({'No':0 , 'Yes':1})

In [7]:
df.dtypes

Date                 str
Location             str
MinTemp          float64
MaxTemp          float64
Rainfall         float64
Evaporation      float64
Sunshine         float64
WindGustDir          str
WindGustSpeed    float64
WindDir9am           str
WindDir3pm           str
WindSpeed9am     float64
WindSpeed3pm     float64
Humidity9am      float64
Humidity3pm      float64
Pressure9am      float64
Pressure3pm      float64
Cloud9am         float64
Cloud3pm         float64
Temp9am          float64
Temp3pm          float64
RainToday          int64
RainTomorrow       int64
dtype: object

## Add Date-based features

In [9]:
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Location', 'Date']).reset_index(drop=True)


def get_season_sh(month):
    if month in [12, 1, 2]:
        return 'Summer'
    elif month in [3, 4, 5]:
        return 'Autumn'
    elif month in [6, 7, 8]:
        return 'Winter'
    else:  # 9, 10, 11
        return 'Spring'


df['season'] = df['Date'].dt.month.apply(get_season_sh)


## To predict in two days the temperatures and rain

In [10]:
g = df.groupby('Location')

gap_to_next = (g['Date'].shift(-1) - df['Date']).dt.days
gap_to_plus2 = (g['Date'].shift(-2) - df['Date']).dt.days

rain_in_2 = g['RainToday'].shift(-2)
df['RainInTwoDays'] = np.where(gap_to_plus2 == 2, rain_in_2, np.nan)

maxtemp_next = g['MaxTemp'].shift(-1)
df['MaxTempTomorrow'] = np.where(gap_to_next == 1, maxtemp_next, np.nan)

maxtemp_in_2 = g['MaxTemp'].shift(-2)
df['MaxTempInTwoDays'] = np.where(gap_to_plus2 == 2, maxtemp_in_2, np.nan)

del g, gap_to_next, gap_to_plus2, rain_in_2, maxtemp_next, maxtemp_in_2

df[['RainTomorrow', 'RainInTwoDays', 'MaxTempTomorrow','MaxTempInTwoDays']].isna().sum()


RainTomorrow           0
RainInTwoDays       2942
MaxTempTomorrow     1804
MaxTempInTwoDays    3234
dtype: int64

### Remove the date since their is no need

In [11]:
df = df.drop(columns='Date')

### Check before saving

In [12]:
round(100*df.isna().sum()/len(df),1)

Location             0.0
MinTemp              0.3
MaxTemp              0.2
Rainfall             0.0
Evaporation         42.4
Sunshine            47.5
WindGustDir          6.5
WindGustSpeed        6.5
WindDir9am           6.9
WindDir3pm           2.6
WindSpeed9am         0.7
WindSpeed3pm         1.8
Humidity9am          1.1
Humidity3pm          2.5
Pressure9am          9.8
Pressure3pm          9.8
Cloud9am            37.4
Cloud3pm            39.8
Temp9am              0.5
Temp3pm              1.9
RainToday            0.0
RainTomorrow         0.0
season               0.0
RainInTwoDays        2.1
MaxTempTomorrow      1.3
MaxTempInTwoDays     2.3
dtype: float64

## Save pre-processed Dataset 

In [13]:
df.to_csv('Australia_Meteo_preprocess_4.csv') 